<a href="https://colab.research.google.com/github/Anusha-Panicker/RAG-Research-Paper-QA/blob/main/RAG_Research_Paper_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install arxiv pymupdf langchain langchain-text-splitters sentence-transformers faiss-cpu groq rank_bm25 ragas datasets langchain-groq langchain-huggingface "langchain-community<0.4.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.

In [ ]:
import arxiv, os, requests, fitz, numpy as np, faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from groq import Groq
from google.colab import userdata

# Papers (skip download if already present)
os.makedirs("papers", exist_ok=True)
if len(os.listdir("papers")) == 0:
    search = arxiv.Search(query='all:"retrieval augmented generation"', max_results=15, sort_by=arxiv.SortCriterion.Relevance)
    client_arxiv = arxiv.Client()
    for i, result in enumerate(client_arxiv.results(search)):
        r = requests.get(result.pdf_url)
        with open(f"papers/paper_{i+1}.pdf", "wb") as f:
            f.write(r.content)
print("Papers ready:", len(os.listdir("papers")))

# Extract text
os.makedirs("extracted_text", exist_ok=True)
for filename in os.listdir("papers"):
    if filename.endswith(".pdf"):
        doc = fitz.open(f"papers/{filename}")
        full_text = "".join(page.get_text() for page in doc)
        doc.close()
        with open(f"extracted_text/{filename.replace('.pdf','.txt')}", "w", encoding="utf-8") as f:
            f.write(full_text)
print("Text extracted")

# Chunk
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
all_chunks, chunk_sources = [], []
for filename in os.listdir("extracted_text"):
    if filename.endswith(".txt"):
        with open(f"extracted_text/{filename}", "r", encoding="utf-8") as f:
            text = f.read()
        for chunk in splitter.split_text(text):
            all_chunks.append(chunk)
            chunk_sources.append(filename)
print("Total chunks:", len(all_chunks))

# Embeddings + FAISS
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(all_chunks, show_progress_bar=True)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings).astype('float32'))
print("FAISS index built:", index.ntotal)

# BM25
tokenized_chunks = [chunk.lower().split() for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)
print("BM25 ready")

# Groq + reranker
client = Groq(api_key=userdata.get('GROQ_API_KEY'))
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("\n✅ Pipeline rebuilt!")

Papers ready: 15
Text extracted
Total chunks: 1634


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/52 [00:00<?, ?it/s]

FAISS index built: 1634
BM25 ready


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


✅ Pipeline rebuilt!


In [ ]:
def hybrid_search(question, top_k=5, candidate_pool=15, max_per_source=2):
    question_embedding = model.encode([question])
    distances, vector_indices = index.search(np.array(question_embedding).astype('float32'), candidate_pool)
    vector_indices = vector_indices[0]
    tokenized_question = question.lower().split()
    bm25_scores = bm25.get_scores(tokenized_question)
    bm25_indices = np.argsort(bm25_scores)[::-1][:candidate_pool]
    combined_scores = {}
    for rank, idx in enumerate(vector_indices):
        combined_scores[idx] = combined_scores.get(idx, 0) + 1 / (rank + 1)
    for rank, idx in enumerate(bm25_indices):
        combined_scores[idx] = combined_scores.get(idx, 0) + 1 / (rank + 1)
    sorted_indices = sorted(combined_scores.keys(), key=lambda i: combined_scores[i], reverse=True)
    final_indices, source_count = [], {}
    for idx in sorted_indices:
        src = chunk_sources[idx]
        if source_count.get(src, 0) < max_per_source:
            final_indices.append(idx)
            source_count[src] = source_count.get(src, 0) + 1
        if len(final_indices) >= top_k:
            break
    return final_indices

def hybrid_search_reranked(question, top_k=5, candidate_pool=15, max_per_source=3):
    candidates = hybrid_search(question, top_k=candidate_pool, candidate_pool=candidate_pool, max_per_source=max_per_source)
    pairs = [[question, all_chunks[idx]] for idx in candidates]
    rerank_scores = reranker.predict(pairs)
    scored = sorted(zip(candidates, rerank_scores), key=lambda x: x[1], reverse=True)
    return [idx for idx, score in scored[:top_k]]

print("Search functions ready!")

Search functions ready!


In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

groq_llm = ChatGroq(api_key=userdata.get('GROQ_API_KEY'), model="openai/gpt-oss-120b")
evaluator_llm = LangchainLLMWrapper(groq_llm)

hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
evaluator_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

print("RAGAS evaluator ready!")

/tmp/ipykernel_18157/4125866628.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(groq_llm)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAGAS evaluator ready!


/tmp/ipykernel_18157/4125866628.py:10: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)


In [ ]:
test_questions = [
    "What is retrieval augmented generation and why is it used?",
    "What techniques do papers use to reduce hallucination in RAG systems?",
    "How does GFM-RAG improve retrieval accuracy?",
    "What evaluation metrics are commonly used for RAG systems?",
    "What are common chunking strategies used in RAG pipelines?"
]

eval_data = {"question": [], "contexts": [], "answer": []}

for q in test_questions:
    top_indices = hybrid_search_reranked(q, top_k=5)
    contexts = [all_chunks[idx] for idx in top_indices]
    context_text = "\n\n---\n\n".join(contexts)

    prompt = f"""Answer the question using ONLY the context below. If the answer isn't in the context, say so.

Context:
{context_text}

Question: {q}

Answer:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )

    eval_data["question"].append(q)
    eval_data["contexts"].append(contexts)
    eval_data["answer"].append(response.choices[0].message.content)
    print("Done:", q)

print("\nAll test questions answered!")

Done: What is retrieval augmented generation and why is it used?
Done: What techniques do papers use to reduce hallucination in RAG systems?
Done: How does GFM-RAG improve retrieval accuracy?
Done: What evaluation metrics are commonly used for RAG systems?
Done: What are common chunking strategies used in RAG pipelines?

All test questions answered!


In [ ]:
from ragas import EvaluationDataset

samples = [
    {
        "user_input": eval_data["question"][i],
        "retrieved_contexts": eval_data["contexts"][i],
        "response": eval_data["answer"][i],
    }
    for i in range(len(eval_data["question"]))
]
dataset = EvaluationDataset.from_list(samples)
print("Dataset built:", len(samples), "samples")

In [ ]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas.run_config import RunConfig
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper

groq_llm = ChatGroq(
    api_key=userdata.get('GROQ_API_KEY'),
    model="openai/gpt-oss-120b",
    max_tokens=4096   # much higher ceiling — faithfulness checking needs room to verify each claim
)

# quick sanity check: confirm max_tokens actually got set
print("max_tokens set to:", groq_llm.max_tokens)

evaluator_llm = LangchainLLMWrapper(groq_llm)

stable_run_config = RunConfig(max_workers=2, timeout=120)

results = evaluate(
    dataset=dataset,
    metrics=[Faithfulness(), ResponseRelevancy(strictness=1)],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=stable_run_config
)
print("STABLE final results (take 2):", results)

/tmp/ipykernel_18157/1945416970.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy
/tmp/ipykernel_18157/1945416970.py:2: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import Faithfulness, ResponseRelevancy


max_tokens set to: 4096


/tmp/ipykernel_18157/1945416970.py:16: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(groq_llm)


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[2]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[3]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[4]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m283wwpneryb0hjgqmjt0a3s` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199483, Requested 954. Please try again in 3m8.784s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
ERROR:ragas.executor:Exception raised in Job[5]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m283wwpneryb0hjgqmjt0a3s` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199465, Requested 943. Please try again in 2m56.256s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/set

STABLE final results (take 2): {'faithfulness': 0.9333, 'answer_relevancy': 0.8488}


In [ ]:
import faiss
import numpy as np

# Cosine similarity = normalize vectors first, then use inner product
def normalize_vectors(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / norms

# normalize your existing embeddings
normalized_embeddings = normalize_vectors(np.array(embeddings).astype('float32'))

# rebuild FAISS index using Inner Product (IP) instead of L2 distance
dimension = normalized_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)   # IP = Inner Product = cosine similarity (once normalized)
index.add(normalized_embeddings)

print("FAISS index rebuilt with cosine similarity. Total vectors:", index.ntotal)

FAISS index rebuilt with cosine similarity. Total vectors: 1634


In [ ]:
def hybrid_search(question, top_k=5, candidate_pool=15, max_per_source=2):
    # normalize the question embedding too, so it matches our normalized index
    question_embedding = model.encode([question])
    question_embedding = normalize_vectors(np.array(question_embedding).astype('float32'))

    distances, vector_indices = index.search(question_embedding, candidate_pool)
    vector_indices = vector_indices[0]

    tokenized_question = question.lower().split()
    bm25_scores = bm25.get_scores(tokenized_question)
    bm25_indices = np.argsort(bm25_scores)[::-1][:candidate_pool]

    combined_scores = {}
    for rank, idx in enumerate(vector_indices):
        combined_scores[idx] = combined_scores.get(idx, 0) + 1 / (rank + 1)
    for rank, idx in enumerate(bm25_indices):
        combined_scores[idx] = combined_scores.get(idx, 0) + 1 / (rank + 1)

    sorted_indices = sorted(combined_scores.keys(), key=lambda i: combined_scores[i], reverse=True)
    final_indices, source_count = [], {}
    for idx in sorted_indices:
        src = chunk_sources[idx]
        if source_count.get(src, 0) < max_per_source:
            final_indices.append(idx)
            source_count[src] = source_count.get(src, 0) + 1
        if len(final_indices) >= top_k:
            break
    return final_indices

print("hybrid_search updated to use cosine similarity!")

hybrid_search updated to use cosine similarity!


In [ ]:
def get_context_for_question(question, top_k=5, small_doc_threshold=25):
    """
    If there are very few chunks total, just use ALL of them as context
    (no point doing retrieval when there's barely anything to search through).
    Otherwise, use our normal hybrid search + re-ranking.
    """
    if len(all_chunks) <= small_doc_threshold:
        print(f"Small document detected ({len(all_chunks)} chunks) — using ALL chunks as context")
        return list(range(len(all_chunks)))  # just return every chunk index
    else:
        return hybrid_search_reranked(question, top_k=top_k)

In [ ]:
def ask_question_final(question, top_k=5, small_doc_threshold=25):
    comparison_words = [" vs ", " versus ", "compare", "difference between"]
    is_comparison = any(word in question.lower() for word in comparison_words)

    # NEW: check if this is a small document first
    if len(all_chunks) <= small_doc_threshold:
        print(f"Small document detected ({len(all_chunks)} chunks) — using ALL chunks as context")
        top_indices = list(range(len(all_chunks)))
    elif is_comparison:
        parts = question.replace("Compare how", "").replace("compare", "").split(" and ")
        sub_questions = [p.strip() + " improve retrieval accuracy?" for p in parts[:2]]
        print("Detected comparison — searching separately for:")
        all_indices = []
        for sub_q in sub_questions:
            print(" -", sub_q)
            indices = hybrid_search_reranked(sub_q, top_k=top_k)
            all_indices.extend(indices)
        seen = set()
        top_indices = [i for i in all_indices if not (i in seen or seen.add(i))]
    else:
        top_indices = hybrid_search_reranked(question, top_k=top_k)

    retrieved_chunks = []
    for idx in top_indices:
        retrieved_chunks.append(all_chunks[idx])
        print("Retrieved from:", chunk_sources[idx])

    context = "\n\n---\n\n".join(retrieved_chunks)

    prompt = f"""Answer the question using ONLY the context below. If the answer isn't in the context, say so.

Context:
{context}

Question: {question}

Answer:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

print("ask_question_final updated with small-document handling!")

ask_question_final updated with small-document handling!


In [ ]:
# Temporarily pretend we have a small document, just to test the branch
original_chunks_backup = all_chunks
all_chunks_test = all_chunks[:10]  # pretend only 10 chunks exist

# quick manual check (not modifying your real all_chunks)
print("If we had only", len(all_chunks_test), "chunks, would trigger small-doc mode:", len(all_chunks_test) <= 25)

If we had only 10 chunks, would trigger small-doc mode: True


In [ ]:
# Cleaned-up version: get_context_for_question now actually gets used, no duplicate logic

def get_context_for_question(question, top_k=5, small_doc_threshold=25):
    if len(all_chunks) <= small_doc_threshold:
        print(f"Small document detected ({len(all_chunks)} chunks) — using ALL chunks as context")
        return list(range(len(all_chunks)))
    else:
        return hybrid_search_reranked(question, top_k=top_k)

def ask_question_final(question, top_k=5, small_doc_threshold=25):
    comparison_words = [" vs ", " versus ", "compare", "difference between"]
    is_comparison = any(word in question.lower() for word in comparison_words)

    if is_comparison and len(all_chunks) > small_doc_threshold:
        # only bother splitting comparisons when there's actually enough content to search
        parts = question.replace("Compare how", "").replace("compare", "").split(" and ")
        sub_questions = [p.strip() + " improve retrieval accuracy?" for p in parts[:2]]
        print("Detected comparison — searching separately for:")
        all_indices = []
        for sub_q in sub_questions:
            print(" -", sub_q)
            indices = hybrid_search_reranked(sub_q, top_k=top_k)
            all_indices.extend(indices)
        seen = set()
        top_indices = [i for i in all_indices if not (i in seen or seen.add(i))]
    else:
        # covers BOTH normal search AND small-document mode in one call
        top_indices = get_context_for_question(question, top_k=top_k, small_doc_threshold=small_doc_threshold)

    retrieved_chunks = []
    for idx in top_indices:
        retrieved_chunks.append(all_chunks[idx])
        print("Retrieved from:", chunk_sources[idx])

    context = "\n\n---\n\n".join(retrieved_chunks)

    prompt = f"""Answer the question using ONLY the context below. If the answer isn't in the context, say so.

Context:
{context}

Question: {question}

Answer:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

print("Cleaned up: get_context_for_question is now actually used, no duplicate logic!")

Cleaned up: get_context_for_question is now actually used, no duplicate logic!


In [ ]:
answer = ask_question_final("What is retrieval augmented generation and why is it used?")
print("\n\nANSWER:\n", answer)

Retrieved from: paper_15.txt
Retrieved from: paper_15.txt
Retrieved from: paper_12.txt
Retrieved from: paper_9.txt
Retrieved from: paper_6.txt


ANSWER:
 Retrieval Augmented Generation (RAG) is a technique that pairs a pre‑trained language model with a retrieval system. The retriever first fetches relevant documents or passages from a defined external knowledge base (often using dense passage retrieval) and then supplies those retrieved texts as additional context to the language model when it generates its output.  

It is used because this combination lets the model produce answers that are more up‑to‑date and factually grounded without having to retrain the entire model. By bringing in external, domain‑specific information at inference time, RAG improves accuracy on tasks such as question answering and other applications that require current or specialized knowledge.
